In [1]:
import pandas as pd

df = pd.read_excel("C:/Users/DELL/Downloads/hotel_daily_booking_data_2024_2025.xlsx",skiprows=1)
df['date'] = pd.to_datetime(df['date'])

In [2]:
df.head()

,date,room_type,bookings,price_inr,occupancy_pct,event_flag,event_name,day_of_week,is_weekend
0,2024-01-01,Deluxe,57,16000,89.7,1,New Year,Monday,0
1,2024-01-01,Suite,14,30650,64.5,1,New Year,Monday,0
2,2024-01-01,Standard,83,11350,86.8,1,New Year,Monday,0
3,2024-01-02,Deluxe,51,12950,81.6,0,NaN,Tuesday,0
4,2024-01-02,Suite,13,24150,61.5,0,NaN,Tuesday,0


In [3]:
#room_counts = df.groupby(["room_type"])['room_type'].count()

In [4]:
#room_counts

In [5]:
print(df.shape)

(2193, 9)


In [6]:
df = df.sort_values(['room_type','date'])

In [7]:
df['month'] = df['date'].dt.month
#df['day'] = df['date'].dt.day
df['week'] = df['date'].dt.isocalendar().week.astype(int)

In [8]:
df = df.sort_values(['room_type', 'date'])

df['lag_1'] = df.groupby('room_type')['bookings'].shift(1)


df['lag_7'] = df.groupby('room_type')['bookings'].shift(7)


df['rolling_7'] =  df.groupby('room_type')['bookings'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())

df['price_lag_1']      = df.groupby('room_type')['price_inr'].shift(1)
df['price_lag_7']      = df.groupby('room_type')['price_inr'].shift(7)
df['price_rolling_7']  = df.groupby('room_type')['price_inr'].transform(lambda x: x.shift(1).rolling(7).mean())


df = df.dropna(subset=['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7'])


In [9]:
#df[(df['room_type'].isin(['Deluxe','Suite'])) & (df['date']<='2024-01-08')]
df[['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7']].isna().sum()

lag_1              0
lag_7              0
rolling_7          0
price_lag_1        0
price_lag_7        0
price_rolling_7    0
dtype: int64

In [10]:
#df.head()

In [11]:
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ['room_type','day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df = pd.get_dummies(df,columns=['event_name'],drop_first=True)

In [12]:
print(encoders['day_of_week'].classes_)
print(encoders['room_type'].classes_)
print(encoders)

['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']
['Deluxe' 'Standard' 'Suite']
{'room_type': LabelEncoder(), 'day_of_week': LabelEncoder()}


In [13]:
for col, le in encoders.items():
    mapping = dict(zip(le.transform(le.classes_), le.classes_))
    print(f"{col}: {mapping}")


room_type: {np.int64(0): 'Deluxe', np.int64(1): 'Standard', np.int64(2): 'Suite'}
day_of_week: {np.int64(0): 'Friday', np.int64(1): 'Monday', np.int64(2): 'Saturday', np.int64(3): 'Sunday', np.int64(4): 'Thursday', np.int64(5): 'Tuesday', np.int64(6): 'Wednesday'}


In [14]:
##rows = []
##for col, le in encoders.items():
   ## for encoded, original in zip(le.transform(le.classes_), le.classes_):
        ##rows.append({'column': col, 'encoded': encoded, 'original': original})

##pd.DataFrame(rows).to_excel("C:/Users/DELL/Downloads/label_encoding_mapping.xlsx", index=False)
#print("Saved ✅")


# ── Inverse transform later ──────────────────────────────────────────────────
print(encoders['room_type'].inverse_transform([0, 1, 2]))    # → ['Deluxe', 'Standard', 'Suite']
print(encoders['day_of_week'].inverse_transform([0, 1, 2,3,4,5,6]))  # → ['Friday', 'Monday', 'Saturday']


['Deluxe' 'Standard' 'Suite']
['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']


In [15]:
#future['room_type'] = encoders['room_type'].inverse_transform(future['room_type'])
#encoders['room_type'].inverse_transform(future['room_type'])


In [16]:
print(df.columns)

Index(['date', 'room_type', 'bookings', 'price_inr', 'occupancy_pct',
       'event_flag', 'day_of_week', 'is_weekend', 'month', 'week', 'lag_1',
       'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7',
       'event_name_Diwali', 'event_name_New Year', 'event_name_Tamil New Year',
       'event_name_Valentine's Day'],
      dtype='object')


In [17]:
df = df.sort_values(['room_type', 'date'])

In [18]:
cutoff_date = (df['date'].max() - pd.Timedelta(days=30))
train_df = df[df['date'] <= cutoff_date]
val_df = df[df['date'] > cutoff_date]

print(cutoff_date)
print(train_df['date'].min(), train_df['date'].max())
print(val_df['date'].min(), val_df['date'].max())

2025-12-01 00:00:00
2024-01-08 00:00:00 2025-12-01 00:00:00
2025-12-02 00:00:00 2025-12-31 00:00:00


In [19]:
# ── Rename columns with spaces/special chars ──────────────────────────────────
rename_map = {
    'event_name_New Year':        'event_name_New_Year',
    'event_name_Tamil New Year':  'event_name_Tamil_New_Year',
    "event_name_Valentine's Day": 'event_name_Valentines_Day'
}
train_df = train_df.rename(columns=rename_map)
val_df   = val_df.rename(columns=rename_map)
df       = df.rename(columns=rename_map)



In [20]:

# Prophet model for each room type
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

prophet_results = {}

# room_type encoding:
# 0=Deluxe, 1=Standard, 2=Suite

regressors = [
    'is_weekend',
    'month',
    'week',
    'price_lag_1',
    'price_lag_7',
    'price_rolling_7',
    'event_name_Diwali',
    'event_name_New_Year',
    'event_name_Tamil_New_Year',
    'event_name_Valentines_Day'
]

for room in sorted(df['room_type'].unique()):

    train_room = train_df[train_df['room_type'] == room].copy()
    val_room   = val_df[val_df['room_type'] == room].copy()

    prophet_train = train_room.rename(
        columns={'date':'ds', 'bookings':'y'}
    )

    m = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False
    )

    for reg in regressors:
        m.add_regressor(reg)

    m.fit(prophet_train[['ds','y'] + regressors])

    future = val_room.rename(columns={'date':'ds'})

    forecast = m.predict(
        future[['ds'] + regressors]
    )

    preds = forecast['yhat'].values

    mae = mean_absolute_error(
        val_room['bookings'],
        preds
    )

    rmse = np.sqrt(
        np.mean((val_room['bookings'].values - preds)**2)
    )

    prophet_results[room] = {
        'model': m,
        'preds': preds,
        'mae': mae,
        'rmse': rmse
    }

    print(f"Room Type {room}")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print("-"*40)


16:34:33 - cmdstanpy - INFO - Chain [1] start processing
16:34:35 - cmdstanpy - INFO - Chain [1] done processing
16:34:35 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
16:34:35 - cmdstanpy - INFO - Chain [1] start processing
16:34:38 - cmdstanpy - INFO - Chain [1] done processing
16:34:38 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'C:\Users\DELL\anaconda3\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=26212 data file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\wt__u0k2.json init=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\pvhuu65j.json output file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\prophet_modelxoy6ni7d\prophet_model-20260612163435.csv method=optimize algorithm=newton iter=10000' failed: 

In [21]:
room = 0

train_room = train_df[train_df['room_type']==room].copy()

prophet_train = train_room.rename(
    columns={'date':'ds','bookings':'y'}
)

m = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True
)

m.add_regressor('is_weekend')

m.fit(
    prophet_train[['ds','y','is_weekend']],
    algorithm='LBFGS'
)

16:35:50 - cmdstanpy - INFO - Chain [1] start processing
16:35:54 - cmdstanpy - INFO - Chain [1] done processing
16:35:54 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
16:35:54 - cmdstanpy - INFO - Chain [1] start processing
16:35:55 - cmdstanpy - INFO - Chain [1] done processing
16:35:55 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'C:\Users\DELL\anaconda3\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=96836 data file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\q3tlm4eo.json init=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\1dmfcewi.json output file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\prophet_model1t9zs1e3\prophet_model-20260612163554.csv method=optimize algorithm=newton iter=10000' failed: 

In [22]:
from prophet import Prophet

room = 0

train_room = train_df[train_df['room_type']==room].copy()

prophet_train = train_room.rename(
    columns={'date':'ds','bookings':'y'}
)

m = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True
)

m.fit(prophet_train[['ds','y']])

16:36:55 - cmdstanpy - INFO - Chain [1] start processing
16:36:56 - cmdstanpy - INFO - Chain [1] done processing
16:36:56 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
16:36:56 - cmdstanpy - INFO - Chain [1] start processing
16:36:58 - cmdstanpy - INFO - Chain [1] done processing
16:36:58 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'C:\Users\DELL\anaconda3\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=68938 data file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\wj9nsu5c.json init=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\x9r3yl88.json output file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\prophet_modelh5xmn98u\prophet_model-20260612163656.csv method=optimize algorithm=newton iter=10000' failed: 

In [23]:
import prophet
import cmdstanpy

print(prophet.__version__)
print(cmdstanpy.__version__)

1.1.7
1.3.0


In [24]:
prophet_train[['ds','y']].dtypes

ds    datetime64[ns]
y              int64
dtype: object

In [27]:
m = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=False
)

m.fit(
    prophet_train[['ds','y']],
    algorithm='BFGS'
)

16:39:56 - cmdstanpy - INFO - Chain [1] start processing
16:39:57 - cmdstanpy - INFO - Chain [1] done processing
16:39:57 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
16:39:57 - cmdstanpy - INFO - Chain [1] start processing
16:39:57 - cmdstanpy - INFO - Chain [1] done processing
16:39:57 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'C:\Users\DELL\anaconda3\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=58059 data file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\1qgvpv4y.json init=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\9fckmxww.json output file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\prophet_modeldp79vyy3\prophet_model-20260612163957.csv method=optimize algorithm=newton iter=10000' failed: 

In [28]:
from prophet import Prophet

m = Prophet()
m.fit(prophet_train[['ds','y']])

16:40:40 - cmdstanpy - INFO - Chain [1] start processing
16:40:42 - cmdstanpy - INFO - Chain [1] done processing
16:40:42 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
16:40:42 - cmdstanpy - INFO - Chain [1] start processing
16:40:43 - cmdstanpy - INFO - Chain [1] done processing
16:40:43 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'C:\Users\DELL\anaconda3\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=51043 data file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\gq40p34f.json init=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\1q60quot.json output file=C:\Users\DELL\AppData\Local\Temp\tmpoq4e5rv4\prophet_modelfoe3vz22\prophet_model-20260612164042.csv method=optimize algorithm=newton iter=10000' failed: 

In [ ]:

# Plot actual vs Prophet forecast for each room type
import matplotlib.pyplot as plt

for room in sorted(df['room_type'].unique()):

    val_room = val_df[val_df['room_type'] == room].copy()

    plt.figure(figsize=(12,5))

    plt.plot(
        val_room['date'],
        val_room['bookings'],
        label='Observed',
        color='red'
    )

    plt.plot(
        val_room['date'],
        prophet_results[room]['preds'],
        label='Prophet Forecast',
        color='blue'
    )

    plt.title(f'Room Type {room} - Prophet Validation Forecast')
    plt.xlabel('Date')
    plt.ylabel('Bookings')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.show()
